In [2]:
from IPython.lib.deepreload import original_reload
!pip install opencv-python

   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/39.0 MB ? eta -:--:--
    --------------------------------------- 0.5/39.0 MB 689.8 kB/s eta 0:00:56
    --------------------------------------- 0.5/39.0 MB 689.8 kB/s eta 0:00:56
    --------------------------------------- 0.8/39.0 MB 776.2 kB/s eta 0:00:50
   - -------------------------------------- 1.0/39.0 MB 840.2 kB/s eta 0:00:46
   - -------------------------------------- 1.3/39.0 MB 895.5 kB/s eta 0:00:43
   - -------------------------------------- 1.6/39.0 MB 941.7 kB/s eta 0:00:40
   - -------------------------------------- 1.8/39.0 MB 997.2 kB/s eta 0:00:38
   -- ------------------------------------- 2.1/39.0 MB 1.0 MB/s eta 0:00:36
   -- --------------


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import cv2
import numpy as np

In [72]:
original_image = cv2.imread("PrebendsEdit.jpg")
#original_image = cv2.imread("Bluebell.JPG")
#original_image = cv2.imread("Cas.jpg")

In [74]:
cv2.imwrite("debug.png", original_image)

True

In [75]:
scales = [original_image]
for i in range(4):
    scales.append(cv2.resize(scales[-1], None, fx=0.5, fy=0.5, interpolation=cv2.INTER_CUBIC))
cv2.imwrite("debug.png", scales[-1])


True

In [76]:
def BlurImages(img):
    #return cv2.bilateralFilter(img, 20, 100, 100)
    blur = cv2.GaussianBlur(img, (21, 21), 4)
    cv2.imwrite("debugBlur.png", blur)
    return blur

def MakeOdd(num):
    if num <= 1:
        print("Kernal Size was too small so set to 3")
        return 3
    return num - num%2 + 1

In [79]:
def FindEdges(sizes, size, pixel_detail_factor, minimum_edge_threshold, diff_from_median_threshold):
    img = sizes[size]
    blurred = BlurImages(img)

    gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)

    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)  # Horizontal edges
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)  # Vertical edges

    # Compute gradient magnitude
    gradient_magnitude = cv2.magnitude(sobelx, sobely)

    # Convert to uint8
    gradient_magnitude = cv2.convertScaleAbs(gradient_magnitude)


    edges = gradient_magnitude
    cv2.imwrite("debugEdges.png", edges)



    adujsted_threshold = minimum_edge_threshold * 255




    median_edges = cv2.medianBlur(edges, MakeOdd(128 // (2**size) + 1))

    colour_mask = np.bitwise_and((edges > median_edges * diff_from_median_threshold), (edges > adujsted_threshold)) * 255

    cv2.imwrite("debugMask.png", colour_mask)

    blurred_edges = cv2.GaussianBlur(edges, (15, 15), 0)


    alt_median_edges = cv2.medianBlur(blurred_edges, 128 // (2**size) + 1)

    alt_colour_mask = np.bitwise_and((blurred_edges > alt_median_edges * diff_from_median_threshold), (blurred_edges > adujsted_threshold)) * 255

    cv2.imwrite("altDebugMask.png", alt_colour_mask)

    cv2.imwrite("alt.png", np.append(img, alt_colour_mask[:, :, np.newaxis], axis=2))
    out = np.append(img, colour_mask[:, :, np.newaxis], axis=2)

    return out

In [80]:
cv2.imwrite("debug.png",
            FindEdges(scales, size=1,
                      pixel_detail_factor = 1, # Use this to adjust the relative information per pixel.
                      # debugEdges.JPG
                      #
                      # Edge Thresholding:
                      minimum_edge_threshold = 0.05,
                      diff_from_median_threshold = 1.5
                      # debugMask
                      )
            )

True

In [54]:
def ImageComponentAnalysis(sizes, size):
    img = sizes[size]
    blurred = cv2.medianBlur(img, 7)



    bin_size = 48


    binned_image = blurred - (blurred)%bin_size# + bin_size//2

    cv2.imwrite("ICA.png", binned_image)




In [55]:
ImageComponentAnalysis(scales, 1)